In [0]:
## my silver CTAS (with %sql)

In [0]:
# read the message that the task named "bronze" left behind
n = dbutils.jobs.taskValues.get(
    taskKey="bronze",       # WHO sent it: the task name
    key="bronze_rows",      # WHICH message: same label as in .set()
    debugValue=0)
print("silver received from bronze:", n)

In [0]:
%sql
-- Rebuild the silver table from bronze.
-- CREATE OR REPLACE = drop the old version, build a new one (safe to rerun).
CREATE OR REPLACE TABLE workspace.dbx_exam.silver_cbs_bevolking AS
SELECT DISTINCT
  trim(RegioS)                                         AS gemeente_code, -- remove spaces
  CAST(substr(Perioden, 1, 4) AS INT)                  AS jaar,          -- '2025MM01' → 2025
  CAST(substr(Perioden, 7, 2) AS INT)                  AS maand,         -- '2025MM01' → 1
  try_cast(BevolkingAanHetBeginVanDePeriode_1 AS INT)  AS bevolking,     -- text → number (NULL if impossible)
  try_cast(LevendGeborenKinderen_2 AS INT)             AS geboren,
  try_cast(Overledenen_3 AS INT)                       AS overleden,
  source_file
   FROM workspace.dbx_exam.bronze_cbs_bevolking_al
WHERE try_cast(BevolkingAanHetBeginVanDePeriode_1 AS INT) IS NOT NULL;  -- skip municipalities that no longer exist

---------------------------------------------------------------------------
AnalysisException                         Traceback (most recent call last)
File <command-5848797012872886>, line 1
----> 1 get_ipython().run_cell_magic('sql', '', "-- Rebuild the silver table from bronze.\n-- CREATE OR REPLACE = drop the old version, build a new one (safe to rerun).\nCREATE OR REPLACE TABLE workspace.dbx_exam.silver_cbs_bevolking AS\nSELECT DISTINCT\n  trim(RegioS)                                         AS gemeente_code, -- remove spaces\n  CAST(substr(Perioden, 1, 4) AS INT)                  AS jaar,          -- '2025MM01' → 2025\n  CAST(substr(Perioden, 7, 2) AS INT)                  AS maand,         -- '2025MM01' → 1\n  try_cast(BevolkingAanHetBeginVanDePeriode_1 AS INT)  AS bevolking,     -- text → number (NULL if impossible)\n  try_cast(LevendGeborenKinderen_2 AS INT)             AS geboren,\n  try_cast(Overledenen_3 AS INT)                       AS overleden,\n  source_file\n   FROM wo